In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv
/kaggle/input/llm-detect-ai-generated-text/train_prompts.csv
/kaggle/input/llm-detect-ai-generated-text/test_essays.csv
/kaggle/input/llm-detect-ai-generated-text/train_essays.csv


In [2]:
import pandas as pd

# Load datasets
train_df = pd.read_csv(
    "/kaggle/input/llm-detect-ai-generated-text/train_essays.csv"
)
train_df_pr = pd.read_csv(
    "/kaggle/input/llm-detect-ai-generated-text/train_prompts.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/llm-detect-ai-generated-text/test_essays.csv"
)
sample_df=pd.read_csv("/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv")

# Check data
print(train_df.shape)
train_df.sample(10)
test_df.head()
train_df_pr.head()
sample_df.head()



(1378, 4)


,id,generated
0,0000aaaa,0.1
1,1111bbbb,0.9
2,2222cccc,0.4


In [3]:
train_df_pr.head()

,prompt_id,prompt_name,instructions,source_text
0,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,1,Does the electoral college work?,Write a letter to your state senator in which ...,# What Is the Electoral College? by the Office...


In [4]:
train_df.sample(10)

,id,prompt_id,text,generated
1338,f837fc25,1,"Dear Senator, The electoral college has existe...",0
84,130199b7,0,Cars have been the main use for transportation...,0
154,1f272063,0,The automobile was made a long time ago to hel...,0
1309,f330a61e,1,"Dear State Senator, The Electoral College is a...",0
960,acc29d2d,1,"State Senator, The Electoral College is not a ...",0
305,3d41c0ca,0,There are many advantages of limiting your car...,0
340,4348748e,1,Dear senatoor of Florida. I am here tooday too...,0
1280,ee182803,1,electoral College would be better than the pop...,0
615,7405b110,0,"Cars, though useful, have negative impacts on ...",0
681,7e657ec1,0,"Places around the world such as Germany, Ameri...",0


In [5]:
train_df.isnull().sum()

id           0
prompt_id    0
text         0
generated    0
dtype: int64

In [6]:
train_df_pr.isnull().sum()

prompt_id       0
prompt_name     0
instructions    0
source_text     0
dtype: int64

In [7]:
train_merged = train_df.merge(
    train_df_pr,
    on="prompt_id",
    how="left"
)

In [8]:
train_merged.head()

,id,prompt_id,text,generated,prompt_name,instructions,source_text
0,0059830c,0,Cars. Cars have been around since they became ...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,005db917,0,Transportation is a large necessity in most co...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
3,00940276,0,How often do you ride in a car? Do you drive a...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."


In [9]:
print(train_merged.shape)
train_merged.head()


(1378, 7)


,id,prompt_id,text,generated,prompt_name,instructions,source_text
0,0059830c,0,Cars. Cars have been around since they became ...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,005db917,0,Transportation is a large necessity in most co...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
3,00940276,0,How often do you ride in a car? Do you drive a...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."


In [10]:
train_merged["full_text"] = (
    train_merged["instructions"].fillna("") + " " +
    train_merged["source_text"].fillna("") + " " +
    train_merged["text"].fillna("")
)


In [11]:
train_merged.head()

,id,prompt_id,text,generated,prompt_name,instructions,source_text,full_text
0,0059830c,0,Cars. Cars have been around since they became ...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ...",Write an explanatory essay to inform fellow ci...
1,005db917,0,Transportation is a large necessity in most co...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ...",Write an explanatory essay to inform fellow ci...
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ...",Write an explanatory essay to inform fellow ci...
3,00940276,0,How often do you ride in a car? Do you drive a...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ...",Write an explanatory essay to inform fellow ci...
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ...",Write an explanatory essay to inform fellow ci...


In [12]:
X= train_merged['full_text']
y=train_merged['generated']

In [13]:
test_merged = test_df.merge(
    train_df_pr,
    on="prompt_id",
    how="left"
)

test_merged["full_text"] = (
    test_merged["instructions"].fillna("") + " " +
    test_merged["source_text"].fillna("") + " " +
    test_merged["text"].fillna("")
)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


In [15]:
X = train_merged["full_text"]
y = train_merged["generated"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [16]:
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=300000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)


In [17]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)


LogisticRegression(max_iter=1000, n_jobs=-1)

In [18]:
# val_probs = model.predict_proba(X_val_tfidf)[:, 1]

val_probs=model.predict_proba(X_val_tfidf)[:,1]

In [19]:
roc_auc = roc_auc_score(y_val, val_probs)
print("Validation ROC-AUC:", roc_auc)


Validation ROC-AUC: 0.9818181818181818


In [20]:
# Train on full training data
X_full = tfidf.fit_transform(train_merged["full_text"])
X_test = tfidf.transform(test_merged["full_text"])

model.fit(X_full, y)

test_probs = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test_merged["id"],
    "generated": test_probs
})

submission.to_csv("submission.csv", index=False)
